# YOLO Pose — training a landing-base detection model (Kaggle)

Before running:
1. Top right: **Session options → Accelerator → GPU T4 x2**
2. Right panel: **Add Data → Upload** → upload the zip with JSON from Label Studio
   Export → **JSON** (not COCO!) → pack the json + the images folder into a single zip

YOLO Pose label format (each line in a .txt):
```
class cx cy w h  kp1x kp1y kp1v  kp2x kp2y kp2v  ...  kp5x kp5y kp5v
```
- `class` = 0 (landing base)
- `cx cy w h` — bbox, normalized 0-1
- `kp1` — landing-base center (x, y, visibility)
- `kp2..kp5` — the 4 corners of the square base, in fixed order TL, TR, BR, BL (x, y, visibility)
- `visibility`: 2 = visible, 1 = partially visible, 0 = not visible / absent

In [ ]:
#!pip install ultralytics==8.0.235 -q  # for yolo8
!pip install -U ultralytics -q  # yolo11
import ultralytics
print("ultralytics version:", ultralytics.__version__)

# Patch for PyTorch 2.6 compatibility
import torch
_orig_load = torch.load
def _patched_load(f, map_location=None, pickle_module=None, **kwargs):
    kwargs['weights_only'] = False
    if pickle_module is not None:
        return _orig_load(f, map_location=map_location, pickle_module=pickle_module, **kwargs)
    return _orig_load(f, map_location=map_location, **kwargs)
torch.load = _patched_load

import ultralytics
print("ultralytics:", ultralytics.__version__)
print("torch:", torch.__version__)


In [ ]:
# 2. Inspect the uploaded dataset
from pathlib import Path

input_root = Path('/kaggle/input')
for p in sorted(input_root.rglob('*'))[:30]:
    print(p)

In [ ]:
# 3. Convert Label Studio JSON -> YOLO Pose
# Read the native Label Studio JSON (not COCO!).
# The bbox is taken from rectanglelabels, accounting for rotation.
# Keypoints: 1 'center' + up to 4 'corner' points (square landing base).
# All corners share the same label 'corner' and are not pre-ordered, so we
# assign each corner to a fixed slot (TL, TR, BR, BL) by its angle around the
# center. This keeps the keypoint order consistent across all images.
# Keypoint order written to the label: center, TL, TR, BR, BL.
# Missing corners are written as "0 0 0" (visibility 0).
import json, shutil, math, numpy as np
from pathlib import Path

# Find the Label Studio annotations JSON
ls_json = next(Path('/kaggle/input').rglob('*.json'))
print('Label Studio JSON:', ls_json)

with open(ls_json) as f:
    tasks = json.load(f)

images_root = Path('/kaggle/input')

out_images = Path('/kaggle/working/dataset_raw/images')
out_labels = Path('/kaggle/working/dataset_raw/labels')
out_images.mkdir(parents=True, exist_ok=True)
out_labels.mkdir(parents=True, exist_ok=True)

def ls_rect_to_yolo_aabb(rect):
    """
    Label Studio stores a rotated rectangle as:
      x, y          = top-left corner of the UNrotated rect (in % of image size)
      width, height = size of the UNrotated rect (in %)
      rotation      = rotation angle around the top-left corner (in degrees)

    Returns an axis-aligned bbox in YOLO format: cx, cy, w, h (normalized 0-1).
    """
    x_pct = rect['x']
    y_pct = rect['y']
    w_pct = rect['width']
    h_pct = rect['height']
    angle = np.radians(rect.get('rotation', 0))

    cos_a = np.cos(angle)
    sin_a = np.sin(angle)

    # 4 corners in local coords (top-left = rotation pivot)
    corners = np.array([[0, 0], [w_pct, 0], [w_pct, h_pct], [0, h_pct]], dtype=float)

    # Rotate around (0,0) and translate by (x_pct, y_pct)
    rx = corners[:, 0] * cos_a - corners[:, 1] * sin_a + x_pct
    ry = corners[:, 0] * sin_a + corners[:, 1] * cos_a + y_pct

    x1, x2 = np.clip(rx.min(), 0, 100), np.clip(rx.max(), 0, 100)
    y1, y2 = np.clip(ry.min(), 0, 100), np.clip(ry.max(), 0, 100)

    cx_norm = (x1 + x2) / 2 / 100
    cy_norm = (y1 + y2) / 2 / 100
    bw_norm = (x2 - x1) / 100
    bh_norm = (y2 - y1) / 100
    return cx_norm, cy_norm, bw_norm, bh_norm

# Canonical corner directions in image coords (y points down): TL, TR, BR, BL.
# Used as reference angles to map each detected corner to a fixed slot.
CANON = [
    math.atan2(-1, -1),  # 0: TL
    math.atan2(-1,  1),  # 1: TR
    math.atan2( 1,  1),  # 2: BR
    math.atan2( 1, -1),  # 3: BL
]

def ang_dist(a, b):
    """Smallest absolute angular distance between two angles (radians)."""
    d = abs(a - b) % (2 * math.pi)
    return min(d, 2 * math.pi - d)

def assign_corners(center, corners):
    """
    Assign each detected corner to one of the 4 fixed slots (TL, TR, BR, BL)
    by its angle around the center, using a greedy unique assignment.
    Returns a list of 4 (x, y, v) tuples; missing slots become (0, 0, 0).
    """
    cx, cy = center
    pairs = []  # (angular_dist, corner_idx, slot_idx)
    for ci, (x, y) in enumerate(corners):
        a = math.atan2(y - cy, x - cx)
        for si, ref in enumerate(CANON):
            pairs.append((ang_dist(a, ref), ci, si))
    pairs.sort()

    slots = [None] * 4
    used_c, used_s = set(), set()
    for _, ci, si in pairs:
        if ci in used_c or si in used_s:
            continue
        slots[si] = corners[ci]
        used_c.add(ci)
        used_s.add(si)

    out = []
    for s in slots:
        out.append((s[0], s[1], 2) if s is not None else (0.0, 0.0, 0))
    return out

skipped = 0
converted = 0

for task in tasks:
    img_path_raw = task.get('data', {}).get('image', '')
    img_name = Path(img_path_raw).name

    src_img = next(images_root.rglob(img_name), None)
    if src_img is None:
        print(f'Not found: {img_name}')
        skipped += 1
        continue

    annotations = task.get('annotations', [])
    if not annotations:
        skipped += 1
        continue

    lines = []
    for ann in annotations:
        result = ann.get('result', [])

        rects = [r for r in result if r.get('type') == 'rectanglelabels']
        kps_center = [r for r in result if r.get('type') == 'keypointlabels'
                      and r['value'].get('keypointlabels', [''])[0] == 'center']
        kps_corner = [r for r in result if r.get('type') == 'keypointlabels'
                      and r['value'].get('keypointlabels', [''])[0] == 'corner']

        # A bbox and the center keypoint are mandatory; corners may be partial.
        if not rects or not kps_center:
            print(f'Incomplete annotation: {img_name} '
                  f'(rect={len(rects)}, center={len(kps_center)}, corner={len(kps_corner)})')
            continue

        cx_norm, cy_norm, bw_norm, bh_norm = ls_rect_to_yolo_aabb(rects[0]['value'])

        # Center keypoint (normalized 0-1)
        c = kps_center[0]['value']
        center = (c['x'] / 100, c['y'] / 100)

        # Corner keypoints (normalized 0-1), mapped to fixed slots TL,TR,BR,BL
        corners = [(r['value']['x'] / 100, r['value']['y'] / 100) for r in kps_corner]
        slots = assign_corners(center, corners)

        parts = [f"0 {cx_norm:.6f} {cy_norm:.6f} {bw_norm:.6f} {bh_norm:.6f}"]
        parts.append(f"{center[0]:.6f} {center[1]:.6f} 2")  # kp1: center
        for x, y, v in slots:                                # kp2..kp5: corners
            parts.append(f"{x:.6f} {y:.6f} {v}")
        lines.append(" ".join(parts))

    if not lines:
        skipped += 1
        continue

    stem = Path(img_name).stem
    (out_labels / f'{stem}.txt').write_text('\n'.join(lines))
    shutil.copy(src_img, out_images / img_name)
    converted += 1

print(f'\nConverted: {converted}, skipped: {skipped}')

In [ ]:
# 4. Split into train/val (70/30)
import random

random.seed(42)

images = sorted((Path('/kaggle/working/dataset_raw/images')).glob('*'))
random.shuffle(images)

split = int(len(images) * 0.7)
train_imgs = images[:split]
val_imgs   = images[split:]

for split_name, imgs in [('train', train_imgs), ('val', val_imgs)]:
    Path(f'/kaggle/working/dataset/images/{split_name}').mkdir(parents=True, exist_ok=True)
    Path(f'/kaggle/working/dataset/labels/{split_name}').mkdir(parents=True, exist_ok=True)
    for img in imgs:
        shutil.copy(img, f'/kaggle/working/dataset/images/{split_name}/{img.name}')
        lbl = Path('/kaggle/working/dataset_raw/labels') / (img.stem + '.txt')
        if lbl.exists():
            shutil.copy(lbl, f'/kaggle/working/dataset/labels/{split_name}/{lbl.name}')

print(f'Train: {len(train_imgs)}, Val: {len(val_imgs)}')

In [ ]:
# 5. Create data.yaml
yaml_content = """path: /kaggle/working/dataset
train: images/train
val: images/val

nc: 1
names: ['base_pouso']

kpt_shape: [5, 3]            # 5 keypoints, 3 values each (x, y, visibility)
flip_idx: [0, 2, 1, 4, 3]   # horizontal flip: center stays, TL<->TR, BR<->BL
"""

with open('/kaggle/working/data.yaml', 'w') as f:
    f.write(yaml_content)

print(yaml_content)

In [ ]:
# 6 - Training
import sys

# Disable the wandb and raytune callbacks
base = '/usr/local/lib/python3.12/dist-packages/ultralytics/utils/callbacks/'
for fname in ('wb.py', 'raytune.py'):
    with open(base + fname, 'w') as f:
        f.write('callbacks = {}\n')

for mod in list(sys.modules.keys()):
    if 'callbacks.wb' in mod or 'callbacks.raytune' in mod:
        del sys.modules[mod]

import os
os.chdir('/kaggle/working')

from ultralytics import YOLO
model = YOLO('yolo11n-pose.pt')

results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=220,
    cos_lr=False,        # enable
    lr0=0.001,           # initial LR
    lrf=0.01,            # final = lr0 * lrf = 0.00001
    imgsz=640,
    batch=64,
    device=[0, 1],
    project='base_pouso',
    name='run1',
    patience=50,
    # Color jitter
    hsv_h=0.015,   # hue variation
    hsv_s=0.7,     # saturation variation
    hsv_v=0.2,     # brightness variation (darken/brighten)

    # Blur
    #blur=2.0,       # random gaussian blur

    # Geometry (keypoints are transformed automatically)
    degrees=10,     # rotations +/-10 deg
    fliplr=0.4,     # horizontal flip (uses flip_idx to swap TL<->TR, BR<->BL)
    flipud=0.2,     # vertical flip
    scale=0.5,      # scaling +/-50%
    translate=0.0,  # shift
    #perspective=0.0005,  # perspective distortion

    # Mosaic (stitching 4 images)
    #mosaic=0.0
    #mosaic_scale=(0.5, 1.5),
)

In [ ]:
# 6.5 mAP-per-epoch plot
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

def get_run_dir():
    """Return RUN_DIR from the current session, or find the most recent run."""
    if 'RUN_DIR' in globals():
        return globals()['RUN_DIR']
    # Fallback: find the most recent folder
    candidates = list(Path('/kaggle/working').rglob('weights/best.pt'))
    if not candidates:
        raise FileNotFoundError("No completed training run found")
    latest = max(candidates, key=lambda p: p.stat().st_mtime)
    return latest.parent.parent

RUN_DIR = get_run_dir()
print(f"Using: {RUN_DIR}")

csv_path = RUN_DIR / 'results.csv'
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# mAP50 and mAP50-95 — bbox detection
axes[0].plot(df['epoch'], df['metrics/mAP50(B)'],   label='mAP50 (box)')
axes[0].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95 (box)')
axes[0].set_title('mAP — Bounding Box')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

# mAP50 and mAP50-95 — pose (keypoints)
axes[1].plot(df['epoch'], df['metrics/mAP50(P)'],    label='mAP50 (pose)')
axes[1].plot(df['epoch'], df['metrics/mAP50-95(P)'], label='mAP50-95 (pose)')
axes[1].set_title('mAP — Pose (Keypoints)')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True)

# Loss
axes[2].plot(df['epoch'], df['train/pose_loss'], label='train pose loss')
axes[2].plot(df['epoch'], df['val/pose_loss'],   label='val pose loss')
axes[2].set_title('Pose Loss')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

print(f"Best mAP50 (pose):    {df['metrics/mAP50(P)'].max():.4f}  (epoch {df['metrics/mAP50(P)'].idxmax()})")
print(f"Best mAP50-95 (pose): {df['metrics/mAP50-95(P)'].max():.4f}  (epoch {df['metrics/mAP50-95(P)'].idxmax()})")

In [ ]:
# 7. Check — landing-base orientation
# Keypoint order: [center, TL, TR, BR, BL]. The base orientation (yaw) is
# estimated from the center to the TL corner. Adjust if you need a different
# reference.
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from pathlib import Path

BEST_PT = RUN_DIR / 'weights' / 'best.pt'
model = YOLO(str(BEST_PT))

val_images = list(Path('/kaggle/working/dataset/images/val').glob('*'))
#results = model(str(val_images[3]))

results_list = []
for i in range(min(20, len(val_images))):
    results_list.append(model(str(val_images[i])))

for results in results_list:
    for r in results:
        if r.keypoints is None or len(r.keypoints) == 0:
            print('Landing base not found')
            continue

        kps = r.keypoints.xy[0].cpu().numpy()  # shape (5, 2): center, TL, TR, BR, BL
        center = kps[0]
        tl     = kps[1]

        dx = tl[0] - center[0]
        dy = tl[1] - center[1]
        angle_deg = (np.degrees(np.arctan2(dy, dx)) - 90) % 360

        print(f'Center: {center}')
        print(f'Corners (TL, TR, BR, BL): {kps[1:5].tolist()}')
        print(f'Landing-base orientation: {angle_deg:.1f} deg')

        # Display via matplotlib (r.show() does not work in 8.0.235)
        img = r.plot()  # returns a BGR numpy array
        plt.figure(figsize=(8, 6))
        plt.imshow(img[:, :, ::-1])  # BGR -> RGB
        plt.axis('off')
        plt.show()

In [ ]:
# 8. The model is here — download it via the Output panel on the right
print(f'Model: {RUN_DIR / "weights" / "best.pt"}')

In [ ]:
!pip install onnxscript
from ultralytics import YOLO
model = YOLO(str(RUN_DIR / 'weights' / 'best.pt'))
#model.export(format='onnx', imgsz=480, opset=12)

# Saved alongside: .../weights/best.onnx

In [ ]:
import requests, zipfile, shutil, os, stat, io

# Get the latest pnnx release
api = requests.get("https://api.github.com/repos/pnnx/pnnx/releases/latest").json()
print("Version:", api['tag_name'])


asset_url = next(a['browser_download_url'] for a in api['assets']
                 if a['name'] == 'pnnx-20260409-linux.zip')
print("URL:", asset_url)

r = requests.get(asset_url)
with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    print("Files in archive:", z.namelist())
    z.extractall("pnnx_x86")

# Locate the binary
#pnnx_bin = next(f for f in os.listdir("pnnx_x86") if 'x86_64' in f or f == 'pnnx')
src = "pnnx_x86/pnnx-20260409-linux/pnnx"
dst = '/usr/local/lib/python3.12/dist-packages/ultralytics/pnnx'
shutil.copy2(src, dst)
os.chmod(dst, stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)

model.export(format='ncnn', imgsz=1024, half=True)

In [ ]:
import requests

api = requests.get("https://api.github.com/repos/pnnx/pnnx/releases/latest").json()
print("Version:", api['tag_name'])
print("Assets:")
for a in api['assets']:
    print(" -", a['name'])